# Week 5 - Capstone Modeling

## 1. Objective

Reproduce a leakage-controlled classification benchmark for the repository's Starter proxy model using the documented client-group holdout. The target describes an observed decline in the current export window. It is not a future forecast.

This notebook uses the local anonymized starter data and the feature, leakage, split, baseline, preprocessing, and metric utilities in src/content_engine. It does not write into data/ or reports/.

## 2. Dataset loading and contract verification

The source-of-truth file is data/raw/content_refresh_anonymized.csv. The documented clean modeling slice is 30,000 pages by 44 columns. The raw file is expected to contain the nine restricted fields as well; those fields are dropped immediately before feature construction.

In [1]:
from pathlib import Path
import sys
import json
import tempfile

import numpy as np
import pandas as pd
import sklearn
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Locate the repository root from the notebook working directory.
BASE_DIR = Path.cwd().resolve()
while BASE_DIR != BASE_DIR.parent and not (BASE_DIR / 'src' / 'content_engine' / 'config.py').exists():
    BASE_DIR = BASE_DIR.parent
if not (BASE_DIR / 'src' / 'content_engine' / 'config.py').exists():
    raise FileNotFoundError('Run this notebook from inside the content-intelligence-engine repository.')
sys.path.insert(0, str(BASE_DIR / 'src'))

from content_engine.config import (
    RAW_DATA_PATH, DOCUMENTED_44_COLUMNS, RESTRICTED_FLYRANK_COLUMNS,
    IDENTIFIER_COLUMNS, TARGET_COLUMN, FORBIDDEN_LEAKAGE_COLUMNS,
    NUMERIC_FEATURES, CATEGORICAL_FEATURES, RANDOM_SEED,
)
from content_engine.features.builder import transform_dataframe_to_features
from content_engine.features.leakage_audit import audit_feature_names
from content_engine.models.train import client_aware_split, create_preprocessor
from content_engine.models.baseline import compute_baseline_score, evaluate_ranking

print(f'Python: {sys.version.split()[0]} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}')
print(f'Dataset path: {RAW_DATA_PATH.relative_to(BASE_DIR)}')

Python: 3.13.13 | pandas: 3.0.5 | scikit-learn: 1.9.1
Dataset path: data\raw\content_refresh_anonymized.csv


In [2]:
raw_df = pd.read_csv(RAW_DATA_PATH)
restricted_found = [column for column in RESTRICTED_FLYRANK_COLUMNS if column in raw_df.columns]
clean_df = raw_df.drop(columns=restricted_found).copy()

assert len(raw_df) == 30_000, f'Unexpected row count: {len(raw_df)}'
assert len(restricted_found) == 9, f'Expected 9 restricted columns, found {len(restricted_found)}'
assert set(clean_df.columns) == set(DOCUMENTED_44_COLUMNS), 'Post-quarantine columns do not match the documented 44-column contract.'
assert not (set(restricted_found) & set(clean_df.columns))
assert clean_df.content_id.is_unique, 'Expected one row per content item.'
assert (clean_df.impressions_90d > 0).all(), 'The documented starter slice requires positive 90-day impressions.'
assert (clean_df.content_age_days >= 90).all(), 'The documented starter slice requires content age of at least 90 days.'
clean_df = clean_df[DOCUMENTED_44_COLUMNS].copy()

print(f'Raw shape: {raw_df.shape}; after restricted-field quarantine: {clean_df.shape}')
print(f'Pages: {clean_df.content_id.nunique():,}; client groups: {clean_df.client_id.nunique()}')
print(f'Restricted fields removed before feature construction: {restricted_found}')
print('Documented 44-column schema verified.')

Raw shape: (30000, 53); after restricted-field quarantine: (30000, 44)
Pages: 30,000; client groups: 32
Restricted fields removed before feature construction: ['health_score', 'needs_indexing', 'is_quick_win', 'needs_ctr_fix', 'needs_engagement_fix', 'ai_opportunity', 'is_underperformer', 'is_declining', 'is_initial_refresh_candidate']
Documented 44-column schema verified.


## 3. Target and label definition

The binary target column is is_declining_label. The source label is trend_direction: the repository feature builder defines the target as trend_direction == down. In the data dictionary, down means the most recent 30-day impressions are more than 20% below the preceding 30-day impressions. This is a current-window proxy derived within one 90-day export, not a future outcome.

In [3]:
assert 'trend_direction' in clean_df.columns
assert TARGET_COLUMN not in clean_df.columns, 'The binary target should be constructed by the existing feature utility.'
feature_df = transform_dataframe_to_features(clean_df)
expected_target = (clean_df['trend_direction'] == 'down').astype(int)
assert feature_df[TARGET_COLUMN].equals(expected_target), 'Target differs from the documented construction.'
target_counts = feature_df[TARGET_COLUMN].value_counts().sort_index()
print(f'Target column: {TARGET_COLUMN}')
print('Positive class 1 = observed trend_direction down; negative class 0 = all other documented directions.')
print(f'Class counts: {target_counts.to_dict()}')
print(f'Positive prevalence: {target_counts.loc[1] / len(feature_df):.4%}')

Target column: is_declining_label
Positive class 1 = observed trend_direction down; negative class 0 = all other documented directions.
Class counts: {0: 13738, 1: 16262}
Positive prevalence: 54.2067%


## 4. Feature selection and leakage prevention

The only candidate features are the centralized NUMERIC_FEATURES and CATEGORICAL_FEATURES lists in content_engine.config. Hashed identifiers are used only for grouping. Restricted product decisions, the target, direct target-construction fields, and any last_30d / prev_30d tokens are excluded. The existing dynamic audit is run against the actual estimator feature names.

In [4]:
MODEL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
assert len(NUMERIC_FEATURES) == 31 and len(CATEGORICAL_FEATURES) == 9
assert len(MODEL_FEATURES) == 40 and len(set(MODEL_FEATURES)) == 40
assert set(MODEL_FEATURES).issubset(feature_df.columns)

with tempfile.TemporaryDirectory(prefix='week5_leakage_audit_') as audit_dir:
    leakage_result = audit_feature_names(MODEL_FEATURES, reports_dir=Path(audit_dir))
assert leakage_result['audit_passed'] and leakage_result['forbidden_features_detected'] == 0
assert not (set(MODEL_FEATURES) & set(RESTRICTED_FLYRANK_COLUMNS))
assert not (set(MODEL_FEATURES) & set(IDENTIFIER_COLUMNS))
assert TARGET_COLUMN not in MODEL_FEATURES

forbidden_tokens = ('last_30d', 'prev_30d', 'trend_pct', 'trend_dir')
assert not any(token in name.lower() for name in MODEL_FEATURES for token in forbidden_tokens)
print(f'Leakage audit passed: {leakage_result["total_features_evaluated"]} estimator inputs; 0 forbidden fields.')
print(f'Numeric features ({len(NUMERIC_FEATURES)}): {NUMERIC_FEATURES}')
print(f'Categorical features ({len(CATEGORICAL_FEATURES)}): {CATEGORICAL_FEATURES}')

Leakage audit passed: 40 estimator inputs; 0 forbidden fields.
Numeric features (31): ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'content_age_days', 'days_since_last_update', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'age_tier_order', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'is_missing_position', 'is_missing_keyword_context', 'is_missing_word_count', 'engaged_session_ratio', 'click_to_session_ratio']
Categorical features (9): ['competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier']


## 5. Existing preprocessing

The repository feature builder handles missingness before model fitting: numeric feature nulls become 0, avg_position==0 is replaced by the median of observed positive positions and is accompanied by is_missing_position, and categorical nulls become the string unknown. It also creates approved missingness indicators, log totals, and ratios. The sklearn ColumnTransformer then standard-scales numeric features and one-hot encodes categorical features with unknown-category tolerance. No imputer is fitted inside the estimator pipeline because the builder supplies complete candidate columns.

In [5]:
X = feature_df[MODEL_FEATURES].copy()
y = feature_df[TARGET_COLUMN].to_numpy()
assert not X.isna().any().any(), 'Feature builder left missing values in estimator inputs.'
print(f'Estimator matrix: {X.shape}; missing candidate values: {int(X.isna().sum().sum())}')
print('Preprocessor: StandardScaler for numeric features; OneHotEncoder(handle_unknown=ignore) for categorical features.')

Estimator matrix: (30000, 40); missing candidate values: 0
Preprocessor: StandardScaler for numeric features; OneHotEncoder(handle_unknown=ignore) for categorical features.


## 6. Validation strategy

Use the repository's fixed client_aware_split: shuffle unique client_id groups with seed 42 and hold out 20% of clients. This preserves the documented 26-client train / 6-client holdout split and prevents pages from a client appearing on both sides. The holdout tests transfer to unseen client groups; it is not a temporal future-window validation.

In [6]:
train_df, holdout_df, train_clients, holdout_clients = client_aware_split(feature_df, seed=RANDOM_SEED)
assert set(train_clients).isdisjoint(holdout_clients)
assert len(train_clients) == 26 and len(holdout_clients) == 6
assert len(train_df) == 26_619 and len(holdout_df) == 3_381

X_train = train_df[MODEL_FEATURES]
y_train = train_df[TARGET_COLUMN].to_numpy()
X_holdout = holdout_df[MODEL_FEATURES]
y_holdout = holdout_df[TARGET_COLUMN].to_numpy()
print(f'Training: {len(train_df):,} rows / {len(train_clients)} client groups')
print(f'Holdout: {len(holdout_df):,} rows / {len(holdout_clients)} client groups')
print(f'Holdout client groups: {holdout_clients}')
print(f'Positive rows: train={int(y_train.sum()):,}; holdout={int(y_holdout.sum()):,}')
print(f'Client overlap: {len(set(train_clients) & set(holdout_clients))}')

Training: 26,619 rows / 26 client groups
Holdout: 3,381 rows / 6 client groups
Holdout client groups: ['client_8b940be7fb', 'client_bbb965ab0c', 'client_9400f1b21c', 'client_a88a7902cb', 'client_8527a891e2', 'client_9f14025af0']
Positive rows: train=14,487; holdout=1,775
Client overlap: 0


## 7. Transparent baseline and candidate models

Evaluate the existing transparent ranking baseline, Logistic Regression, Random Forest, and Gradient Boosting. Model definitions and parameters match content_engine.models.train. The binary Precision, Recall, and F1 values use the repository evaluator's 0.5 threshold; Precision@K ranks holdout scores and measures the share of positives in the first K pages.

In [7]:
results = {'transparent_baseline': evaluate_ranking(y_holdout, compute_baseline_score(holdout_df).to_numpy())}
models = {
    'logistic_regression': Pipeline([
        ('preprocessor', create_preprocessor()),
        ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)),
    ]),
    'random_forest': Pipeline([
        ('preprocessor', create_preprocessor()),
        ('classifier', RandomForestClassifier(n_estimators=100, max_depth=12, random_state=RANDOM_SEED, n_jobs=-1)),
    ]),
    'gradient_boosting': Pipeline([
        ('preprocessor', create_preprocessor()),
        ('classifier', GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=RANDOM_SEED)),
    ]),
}

for model_name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    positive_scores = pipeline.predict_proba(X_holdout)[:, 1]
    results[model_name] = evaluate_ranking(y_holdout, positive_scores)
    print(f'Fitted {model_name}.')

# Retain the source trainer's project-level selection rule for faithful comparison.
selected_model = 'random_forest'
if results['gradient_boosting']['precision_at_50'] > results['random_forest']['precision_at_50']:
    selected_model = 'gradient_boosting'
elif results['gradient_boosting']['precision_at_50'] == results['random_forest']['precision_at_50']:
    if results['gradient_boosting']['pr_auc'] > results['random_forest']['pr_auc']:
        selected_model = 'gradient_boosting'
print(f"Existing source selection rule result: {selected_model}")

Fitted logistic_regression.
Fitted random_forest.
Fitted gradient_boosting.
Existing source selection rule result: gradient_boosting


## 8. Holdout results and comparison

The result table is generated from this execution, not manually entered. All metrics use only the client-group holdout. N/A is not needed because every listed metric is calculated by the existing evaluator.

In [8]:
metric_columns = [
    ('roc_auc', 'ROC-AUC'), ('pr_auc', 'PR-AUC'), ('precision', 'Precision'),
    ('recall', 'Recall'), ('f1', 'F1'), ('precision_at_20', 'P@20'),
    ('precision_at_50', 'P@50'), ('precision_at_100', 'P@100'),
]
print('| Model | ' + ' | '.join(label for _, label in metric_columns) + ' |')
print('|---|' + '|'.join(['---:'] * len(metric_columns)) + '|')
for model_name, metrics in results.items():
    values = ' | '.join(f'{metrics[key]:.4f}' for key, _ in metric_columns)
    print(f'| {model_name.replace("_", " ")} | {values} |')

assert set(results) == {'transparent_baseline', 'logistic_regression', 'random_forest', 'gradient_boosting'}
print(f'\nAll candidate runs completed. Project-level selected model: {selected_model}.')

| Model | ROC-AUC | PR-AUC | Precision | Recall | F1 | P@20 | P@50 | P@100 |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| transparent baseline | 0.5938 | 0.5669 | 0.4600 | 0.0777 | 0.1330 | 0.3500 | 0.3600 | 0.3300 |
| logistic regression | 0.6510 | 0.6428 | 0.6109 | 0.7324 | 0.6662 | 0.7500 | 0.7000 | 0.7000 |
| random forest | 0.6617 | 0.6287 | 0.6095 | 0.7915 | 0.6887 | 0.4500 | 0.4400 | 0.4800 |
| gradient boosting | 0.6945 | 0.6791 | 0.6400 | 0.7470 | 0.6894 | 0.6000 | 0.6800 | 0.7200 |

All candidate runs completed. Project-level selected model: gradient_boosting.


## Historical guide results (kept separate)

docs/ml-intern-dataset-and-lane-guide.md contains a separate historical starter-pipeline benchmark with a Decision Tree. Those figures are not the current ContentSignal benchmark below and are not rerun here: the referenced scripts/01-05, outputs, and work/ notebook skeletons are absent from this workspace. The current benchmark is the executable src/content_engine pipeline and is reproduced by this notebook.

## 9. Interpretation

Compare metrics against the review capacity: Logistic Regression has the strongest P@20 and P@50, while Gradient Boosting has the best ROC-AUC, PR-AUC, and P@100 in this holdout. The existing source selection rule compares Gradient Boosting with Random Forest on P@50 (then PR-AUC on a tie), so it selects Gradient Boosting; it does not compare the Logistic Regression result when choosing that project-level champion. Treat the champion label as the repository's historical selection, not a universal winner for every queue size.

The model estimates association with the observed proxy label. Feature importance and probabilities are not causal explanations and do not establish that an editorial change will recover traffic.

In [9]:
gb_preprocessor = models['gradient_boosting'].named_steps['preprocessor']
encoded_categories = list(gb_preprocessor.named_transformers_['cat'].get_feature_names_out(CATEGORICAL_FEATURES))
transformed_names = NUMERIC_FEATURES + encoded_categories
gb_importances = models['gradient_boosting'].named_steps['classifier'].feature_importances_
top_indices = np.argsort(gb_importances)[::-1][:10]
print('Top 10 Gradient Boosting impurity importances (associations, not causal effects):')
for rank, index in enumerate(top_indices, start=1):
    print(f'{rank}. {transformed_names[index]}: {gb_importances[index]:.4f}')

Top 10 Gradient Boosting impurity importances (associations, not causal effects):
1. days_with_impressions: 0.3187
2. content_age_days: 0.1791
3. avg_position: 0.0871
4. ctr: 0.0469
5. scroll_rate: 0.0411
6. word_count: 0.0329
7. log_impressions_90d: 0.0258
8. days_with_sessions: 0.0245
9. impressions_90d: 0.0237
10. char_count: 0.0233


## 10. Limitations and reproducibility

- The Starter proxy measures a within-export 30-day impression change; it does not predict a future 30-day outcome. The future-window extension remains unrun and requires warehouse access.
- Validation holds out six client groups from this 30,000-row slice. It does not establish performance on all clients, later periods, or production traffic.
- The repository feature builder's click_to_session_ratio implementation computes sessions divided by clicks (clipped), while prose elsewhere describes clicks divided by sessions. This notebook preserves the existing source behavior so the documented benchmark can be reproduced; treat that feature's name/meaning mismatch as an implementation limitation.
- The data dictionary and capstone report differ by one positive label in their corpus counts. This run verifies the raw data directly: 16,262 positive and 13,738 negative rows.
- The raw CSV contains nine restricted decision columns despite docs/DATA_USE.md saying those product flags are not included. They are dropped before feature construction; the estimator audit confirms none reach the models.

Reproduce with Python and the pinned requirements.txt environment, from a Jupyter kernel opened within this repository. The notebook uses random seed 42, no external downloads, no credentials, and no row-level output. It only reads the local raw CSV and keeps modeling results in memory.